# Formatos avanzados: Parquet y por qué cambia las reglas del juego


## [01] Elige qué columnas leer: la ventaja de Parquet

La tabla "pedidos" tiene 10 columnas: id, fecha, cliente, producto, importe, pais, metodo_pago, direccion, email, notas. Para cada pregunta, escribe SOLO las columnas imprescindibles.

Resultado esperado

- OK    Total facturado por país
- OK    Cuántos pedidos por método de pago
- OK    Ticket medio por mes
- OK    Los 10 clientes que más gastan

Aciertos: 4/4

In [55]:
# La tabla "pedidos" está en Parquet con estas columnas:
#
#   id · fecha · cliente · producto · importe · pais · metodo_pago · direccion · email · notas
#
# Para cada pregunta, escribe la lista de columnas que hay que leer.
# Sólo las imprescindibles: cada columna de más es tiempo de más.

consultas = {
    "Total facturado por país":           ['pais', 'importe'],
    "Cuántos pedidos por método de pago": ['metodo_pago',],
    "Ticket medio por mes":               ['importe', 'fecha'],
    "Los 10 clientes que más gastan":     ['cliente', 'importe'],
}

In [56]:
# --- No toques: la comprobación.
#     La clave va del revés para que no se lea de un vistazo.
#     El orden en que escribas las columnas da igual. ---
CLAVE = {
    "Total facturado por país":           "etropmi,siap",
    "Cuántos pedidos por método de pago": "ogap_odotem",
    "Ticket medio por mes":               "etropmi,ahcef",
    "Los 10 clientes que más gastan":     "etropmi,etneilc",
}

aciertos = 0
for pregunta, clave in CLAVE.items():
    buenas = set(clave[::-1].split(","))
    mias = set(consultas[pregunta])

    if mias == buenas:
        print(f"  OK    {pregunta}")
        aciertos += 1
    elif mias > buenas:
        print(f"  FALLO {pregunta}: lees columnas que no necesitas")
    else:
        print(f"  FALLO {pregunta}: te faltan columnas o no son ésas")
print(f"\nAciertos: {aciertos}/{len(CLAVE)}")

  OK    Total facturado por país
  OK    Cuántos pedidos por método de pago
  OK    Ticket medio por mes
  OK    Los 10 clientes que más gastan

Aciertos: 4/4


## [02] Convierte un CSV de ventas a Parquet y compara tamaños

Genera un CSV con 50.000 registros de ventas, conviértelo a Parquet con pandas y muestra la diferencia de tamaño.

Resultado esperado

- CSV:     1.9 MB
- Parquet: 0.8 MB
  
Parquet ocupa menos de la mitad: True

In [57]:
import csv, os, random
import pandas as pd

random.seed(123)


In [58]:
# Generar CSV con 50,000 ventas
with open("ventas.csv", 'w', newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "producto", "importe", "ciudad", "fecha"])

    for i in range(50_000):
        writer.writerow([
            i+1,
            random.choice(['laptop', "mouse", "teclado", "monitor", "webcam"]),
            round(random.uniform(10, 500), 2),
            random.choice(["madrid", "barcelona", "valencia"]),
            f"2026-{random.randint(1, 12):02d}-{random.randint(1,28):02d}"
        ])

In [59]:
df = pd.read_csv("ventas.csv")
df.to_parquet("ventas.parquet")

In [60]:
# No toques: compara los dos ficheros
tam_csv = os.path.getsize("ventas.csv") / 1024
tam_pq = os.path.getsize("ventas.parquet") / 1024 if os.path.exists("ventas.parquet") else 0
# En MB y con UN decimal a propósito: el tamaño exacto del Parquet depende de
# la versión de la librería que lo escribe, y así la comparación aguanta esa
# variación en vez de suspenderte por unos KB de diferencia.
print(f"CSV:     {tam_csv/1024:.1f} MB")
print(f"Parquet: {tam_pq/1024:.1f} MB")   # si sale 0.0, el fichero aún no existe
print(f"Parquet ocupa menos de la mitad: {0 < tam_pq < tam_csv / 2}")

CSV:     1.9 MB
Parquet: 0.6 MB
Parquet ocupa menos de la mitad: True


## [03] Inspeccionar metadatos de un Parquet

Lee los metadatos de un archivo Parquet sin cargar los datos. El tipo de columna puede salir "string" o "large_string" según tu versión de pyarrow: ambas son correctas.


In [61]:
import pyarrow.parquet as pq

metadata = pq.read_metadata("ventas.parquet")

filas = metadata.num_rows
columnas = metadata.num_columns

In [62]:
# No toques
print("=== Información del archivo Parquet ===")
print(f"Filas: {filas:,}")
print(f"Columnas: {columnas}")
print(f"Row groups: {metadata.num_row_groups}")

schema = pq.read_schema("ventas.parquet")
print("\n=== Esquema (tipos de cada columna) ===")
for i in range(len(schema)):
    print(f"  {schema.field(i).name:15} → {schema.field(i).type}")

=== Información del archivo Parquet ===
Filas: 50,000
Columnas: 5
Row groups: 1

=== Esquema (tipos de cada columna) ===
  id              → int64
  producto        → large_string
  importe         → double
  ciudad          → large_string
  fecha           → large_string


## [04] Column pruning: lee solo lo que necesitas


El CEO quiere el total de ventas por producto. Lee SOLO las columnas necesarias del Parquet (no todas) y calcula el resultado.

Resultado esperado

Columnas leídas: ['importe', 'producto']

Ventas por producto:
- Laptop     → 2,561,133.98€
- Webcam     → 2,560,023.55€
- Monitor    → 2,550,704.75€
- Mouse      → 2,538,846.91€
- Teclado    → 2,516,534.86€


In [63]:
# Solo necesitas dos de las cinco columnas. Ahora mismo pone None
# que para pandas significa "léelo TODO".
COLUMNAS = ['producto', 'importe']      # <- ¿qué dos columnas hacen falta?

df = pd.read_parquet("ventas.parquet", columns=COLUMNAS)

In [64]:
# Calcula total por producto. groupby agrupa filas iguales y .sum() las suma.
totales = df.groupby("producto")['importe'].sum().sort_values(ascending=False)



In [65]:
totales, df.head()

(producto
 laptop     2561133.98
 webcam     2560023.55
 monitor    2550704.75
 mouse      2538846.91
 teclado    2516534.86
 Name: importe, dtype: float64,
   producto  importe
 0   laptop   141.16
 1   laptop   195.77
 2   laptop    88.23
 3    mouse    90.25
 4  monitor    44.24)

In [67]:
# No toques
# sorted() y no list(): pd.read_parquet devuelve las columnas en el orden en que
# las pides, y pedir ["importe","producto"] es igual de correcto. Ordenarlas
# evita suspender a quien las escribe al reves.
print(f"Columnas leídas: {sorted(df.columns)}")
print("Ventas por producto:")
for prod, total in totales.items():
    print(f"  {prod:10} → {total:>12,.2f}€")

Columnas leídas: ['importe', 'producto']
Ventas por producto:
  laptop     → 2,561,133.98€
  webcam     → 2,560,023.55€
  monitor    → 2,550,704.75€
  mouse      → 2,538,846.91€
  teclado    → 2,516,534.86€


## [05] Benchmark: mide la velocidad CSV vs Parquet


Compara el tiempo de leer una columna de CSV (leyendo todo) vs Parquet (column pruning). Los tiempos dependen de tu máquina.

Resultado esperado

- CSV total:     12,727,244.05€
- Parquet total: 12,727,244.05€
- Mismo resultado: True
- Parquet más rápido: True


In [68]:
import time, csv

In [69]:
# --- Método 1: CSV (tiene que leer TODO el fichero) ---
# perf_counter es el cronómetro bueno: no le afectan los cambios de hora.
inicio = time.perf_counter()
total_csv = 0

with open("ventas.csv", 'r') as f:
    reader = csv.DictReader(f)

    for fila in reader:
        total_csv += float(fila['importe'])

tiempo_csv = time.perf_counter() - inicio

In [71]:
# --- Método 2: Parquet (lee SOLO la columna "importe") ---
inicio = time.perf_counter()

df = pd.read_parquet("ventas.parquet", columns=['importe'])

total_pq = df['importe'].sum()
tiempo_parquet = time.perf_counter() - inicio

In [72]:
# No toques
mismo = abs(total_csv - total_pq) < 0.01
# Los milisegundos NO se imprimen, y es a propósito: varían de una ejecución a
# otra en el mismo ordenador, así que compararlos con un texto fijo suspendería
# a quien lo hace bien. Lo que se comprueba es el veredicto: que sale el mismo
# total y que leer una sola columna gana.
print(f"CSV total:     {total_csv:>13,.2f}€")
print(f"Parquet total: {total_pq:>13,.2f}€")
print(f"Mismo resultado: {mismo}")
print(f"Parquet más rápido: {tiempo_parquet < tiempo_csv}")

CSV total:     12,727,244.05€
Parquet total: 12,727,244.05€
Mismo resultado: True
Parquet más rápido: True
